In [1]:
import scanpy as sc
import pandas as pd
import numpy as np  

In [2]:
adata_st = sc.read_h5ad('./visium.h5ad')
adata_sc = sc.read_h5ad('../P4_P6_Tumor.h5ad')

In [3]:
print(adata_sc.obs["level1_celltype"].value_counts()["Multiplet"])
adata_sc = adata_sc[adata_sc.obs["level1_celltype"] != "Multiplet"].copy()

563


In [4]:
celltype_map = {
    "CD1C": "Dendritic cell",
    "Epithelial": "Epithelial cell",
    "Melanocyte": "Tumor cell",
    "Tcell": "T cell",
    "LC": "Dendritic cell",
    "CLEC9A": "Dendritic cell",
    "Mac": "Macrophage",
    "ASDC": "Dendritic cell",
    "B Cell": "B cell / Plasma",
    "PDC": "Dendritic cell",
    "Endothelial Cell": "Endothelial cell",
    "Fibroblast": "Fibroblast",
    "MDSC": "Myeloid cell / MDSC",
    "NK": "NK cell"
}

adata_sc.obs["celltype"] = adata_sc.obs["level1_celltype"].replace(celltype_map)

/tmp/ipykernel_271128/2597024223.py:18: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  adata_sc.obs["celltype"] = adata_sc.obs["level1_celltype"].replace(celltype_map)


In [5]:
sc.tl.rank_genes_groups(adata_sc, groupby='celltype', n_genes=100)
markers_df = pd.DataFrame(adata_sc.uns['rank_genes_groups']['names'])
marker_genes = pd.unique(markers_df.values.ravel()).tolist()

In [6]:
# import matplotlib.pyplot as plt
# fig, axs = plt.subplots(1, 2, figsize=(20, 5))
# sc.pl.spatial(
#     adata_st, color="cluster", alpha=0.7, frameon=False, show=False, ax=axs[0]
# )

# sc.pl.umap(
#     adata_sc, color="level1_celltype", size=10, frameon=False, show=False, ax=axs[1]
# )
# plt.tight_layout()

In [7]:
import tangram as tg

tg.pp_adatas(adata_sc, adata_st, genes=marker_genes)
# 这玩意是把单细胞的细胞映射到spot上，就可以根据这个算spot的细胞类型
ad_map = tg.map_cells_to_space(adata_sc, adata_st,
    mode="cells",
    # mode="clusters",
    # cluster_label='level1_celltype',
    density_prior='rna_count_based',
    num_epochs=500,
    device='cpu'
)

INFO:root:741 training genes are saved in `uns``training_genes` of both single cell and spatial Anndatas.
INFO:root:11765 overlapped genes are saved in `uns``overlap_genes` of both single cell and spatial Anndatas.
INFO:root:uniform based density prior is calculated and saved in `obs``uniform_density` of the spatial Anndata.
INFO:root:rna count based density prior is calculated and saved in `obs``rna_count_based_density` of the spatial Anndata.
INFO:root:Allocate tensors for mapping.
INFO:root:Begin training with 741 genes and rna_count_based density_prior in cells mode...
INFO:root:Printing scores every 100 epochs.


Score: 0.562, KL reg: 0.016
Score: 0.750, KL reg: 0.002
Score: 0.754, KL reg: 0.002
Score: 0.755, KL reg: 0.002
Score: 0.755, KL reg: 0.002


INFO:root:Saving results..


In [8]:
adata_sc.obs

,nCount_RNA,nFeature_RNA,patient,tum.norm,level1_celltype,level2_celltype,level3_celltype,leiden,celltype
P4_Tumor_AAACCTGAGAGAGCTC,7123,2025,P4,Tumor,CD1C,CD1C,CD1C,2,Dendritic cell
P4_Tumor_AAACCTGAGCCCAACC,5558,1953,P4,Tumor,Epithelial,Keratinocyte,Keratinocyte,1,Epithelial cell
P4_Tumor_AAACCTGAGTACATGA,7704,2326,P4,Tumor,Epithelial,Keratinocyte,Keratinocyte,3,Epithelial cell
P4_Tumor_AAACCTGCAATGGACG,5299,1683,P4,Tumor,Epithelial,Keratinocyte,Keratinocyte,5,Epithelial cell
P4_Tumor_AAACCTGCACCATCCT,12485,3017,P4,Tumor,Epithelial,Keratinocyte,Keratinocyte,0,Epithelial cell
...,...,...,...,...,...,...,...,...,...
P6_Tumor_TTTGTCAAGCTATGCT,18941,3780,P6,Tumor,Epithelial,Tumor_KC_Diff,Tumor_KC_Diff,4,Epithelial cell
P6_Tumor_TTTGTCAAGTGAATTG,4670,2093,P6,Tumor,Tcell,Tcell,CD8_Exh,16,T cell
P6_Tumor_TTTGTCATCCAGAGGA,11666,2453,P6,Tumor,B Cell,B Cell,B Cell,17,B cell / Plasma
P6_Tumor_TTTGTCATCCCGGATG,13904,3175,P6,Tumor,CD1C,CD1C,CD1C,7,Dendritic cell


In [9]:
import pandas as pd
import numpy as np

weights = ad_map.X  # 每一行是一个 cell 的空间分布
weights_normalized = weights / weights.sum(axis=1, keepdims=True) * 100 

cell_types = adata_sc.obs.loc[ad_map.obs_names, "celltype"]

weights_df = pd.DataFrame(weights_normalized, index=cell_types, columns=ad_map.var_names)
celltype_composition = weights_df.groupby(level=0).sum().T
celltype_composition_percent = celltype_composition.div(celltype_composition.sum(axis=1), axis=0) * 100
celltype_composition_percent = celltype_composition_percent.round(2)

In [10]:
adata_st.obsm["celltype_composition_percent"] = celltype_composition_percent.loc[adata_st.obs_names].values
adata_st.uns["celltype_composition_percent"] = celltype_composition_percent.loc[adata_st.obs_names]


In [11]:
check = adata_st.uns["celltype_composition_percent"]

In [12]:
adata_st.write_h5ad('./adata_st_anno.h5ad')

In [ ]:
import matplotlib.pyplot as plt

plt.bar(celltype_composition_percent.columns, celltype_composition_percent.loc["GSE144239_GSM4565823_AAACAAGTATCTCCCA-1"])
plt.xticks(rotation=90)
plt.ylabel("Percent of cell mapped to spot")
plt.title("Distribution of cell across spatial spots")
plt.tight_layout()
plt.show()


In [ ]:
tg.project_cell_annotations(ad_map, adata_st, annotation="level1_celltype")
annotation_list = list(pd.unique(adata_sc.obs['level1_celltype']))
tg.plot_cell_annotation_sc(adata_st, annotation_list,perc=0.02)

In [ ]:
tg.plot_training_scores(ad_map, bins=10, alpha=.5)

In [ ]:
ad_map.uns['train_genes_df']

In [ ]:
# 这玩意是把单细胞的表达值映射到spot
ad_ge = tg.project_genes(adata_map=ad_map, adata_sc=adata_sc)
print(ad_ge)

In [ ]:
genes = ['tnfsf13b', 'mnda', 'nr4a2']
ad_map.uns['train_genes_df'].loc[genes]

In [ ]:
tg.plot_genes_sc(genes, adata_measured=adata_st, adata_predicted=ad_ge, perc=0.02)

In [ ]:
ad_ge.write_h5ad('ad_ge.h5ad')
ad_map.write_h5ad('ad_map.h5ad')
adata_sc.write_h5ad('adata_sc.h5ad')
adata_st.write_h5ad('adata_st.h5ad')

In [ ]:
ad_ge = sc.read_h5ad('ad_ge.h5ad')
ad_map = sc.read_h5ad('ad_map.h5ad')
adata_sc = sc.read_h5ad('adata_sc.h5ad')
adata_st = sc.read_h5ad('adata_st.h5ad')

In [ ]:
import seaborn as sns
import tangram as tg
df_all_genes = tg.compare_spatial_geneexp(ad_ge, adata_st, adata_sc)
# sns.scatterplot(data=df_all_genes, x='score', y='sparsity_sp', hue='is_training', alpha=0.5)

tg.plot_auc(df_all_genes)